# SeaDronesSee FP32 Baseline - ConvNeXt backbone + default Faster R-CNN style settings

Notebook nay phuc vu **Giai doan 3: Baseline tren SeaDronesSee**.

Muc tieu:

- tao moc FP32 baseline ban dau de so sanh voi toan bo cac phuong an toi uu ve sau;
- giu co dinh train / validation / test;
- huan luyen voi backbone `ConvNeXt-Tiny`, nhung detector dung cau hinh gan voi **mac dinh Faster R-CNN**;
- chon checkpoint theo validation;
- luu checkpoint tung epoch thanh version dataset Kaggle;
- luu day du metrics, benchmark va anh minh hoa.

Gia dinh baseline trong notebook nay:

- backbone van la `ConvNeXt-Tiny + FPN` de phu hop huong nghien cuu cua repo;
- anchor dung kieu mac dinh Faster R-CNN: `[32, 64, 128, 256, 512]` va aspect ratio `[0.5, 1.0, 2.0]`;
- tat `Focal Loss`, quay ve classification loss mac dinh cua Faster R-CNN;
- kich thuoc anh va optimizer chuyen ve cau hinh baseline gan voi torchvision Faster R-CNN.

Noi dung can luu:

- cau hinh baseline;
- mAP, AP theo lop;
- AP small / medium / large;
- RPN recall;
- FPS, latency;
- RAM, VRAM, kich thuoc model;
- anh dung, bo sot, phat hien nham;
- checkpoint baseline duoc bao toan lam moc so sanh.

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
import time
from base64 import b64encode
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

WORK = Path('/kaggle/working/seadronessee_fp32_baseline')
OUTPUT = WORK / 'checkpoints'
LOGS = WORK / 'logs'
REPORTS = WORK / 'reports'
VISUALS = WORK / 'visuals'

for path in (WORK, OUTPUT, LOGS, REPORTS, VISUALS):
    path.mkdir(parents=True, exist_ok=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}:', torch.cuda.get_device_name(index))
print('Work dir:', WORK)

## 1. Clone repo va cai dependency

In [ ]:
REPO = Path('/kaggle/working/EchteAI')
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'

if not REPO.exists():
    print(f'Cloning {REPO_URL} -> {REPO}', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, cwd='/kaggle/working')
else:
    print(f'Repo already exists: {REPO}', flush=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]',
    'kagglehub', 'psutil', 'matplotlib', 'pandas', 'pillow'
], check=True, cwd=REPO)

sys.path.insert(0, str(REPO))

import yaml
import kagglehub

from pipelines.convnext_qat.checkpoint import checkpoint_size_mb, load_checkpoint, model_state_size_mb
from pipelines.convnext_qat.config import choose_device, load_config
from pipelines.convnext_qat.data import build_coco_loader, unwrap_coco_dataset
from pipelines.convnext_qat.metrics import _coco_metrics, evaluate_model, native_detection_metrics
from pipelines.convnext_qat.models import build_fasterrcnn_convnext

print('Repository ready:', REPO)

## 2. Tim dataset SeaDronesSee va khai bao dataset upload

In [ ]:
SEADRONESSEE_DATASET_HANDLE = 'ubiratanfilho/sds-dataset'


def is_valid_seadronessee_root(root):
    root = Path(root)
    return (
        (root / 'annotations/instances_train.json').exists()
        and (root / 'annotations/instances_val.json').exists()
        and (root / 'images/train').is_dir()
        and (root / 'images/val').is_dir()
    )


def roots_from_annotation(ann_path):
    ann_path = Path(ann_path)
    parents = [ann_path.parent.parent, ann_path.parent.parent.parent, ann_path.parent.parent.parent.parent]
    return [parent for parent in parents if str(parent) != '.']


def find_seadronessee_root(preferred=None, try_download=True):
    candidates = []
    if preferred is not None:
        candidates.append(Path(preferred))
    candidates.extend([
        Path('/kaggle/input/datasets/nguyenducthangtb/seadronessee-compressed'),
        Path('/kaggle/input/seadronessee-compressed'),
        Path('/kaggle/input/sds-dataset/compressed'),
        Path('/kaggle/input/ubiratanfilho/sds-dataset/compressed'),
        Path('/kaggle/input/ubiratanfilho/sds-dataset'),
    ])
    for root in candidates:
        if is_valid_seadronessee_root(root):
            return root
    for ann_path in Path('/kaggle/input').rglob('instances_train.json'):
        for root in roots_from_annotation(ann_path):
            if is_valid_seadronessee_root(root):
                return root
    if try_download:
        print(f'SeaDronesSee not found in /kaggle/input; downloading via kagglehub: {SEADRONESSEE_DATASET_HANDLE}', flush=True)
        download_root = Path(kagglehub.dataset_download(SEADRONESSEE_DATASET_HANDLE))
        if is_valid_seadronessee_root(download_root):
            return download_root
        for ann_path in download_root.rglob('instances_train.json'):
            for root in roots_from_annotation(ann_path):
                if is_valid_seadronessee_root(root):
                    return root
    available = sorted(str(path) for path in Path('/kaggle/input').rglob('instances_train.json'))[:20]
    raise FileNotFoundError(
        'Khong tim thay SeaDronesSee dataset. '
        f'instances_train.json trong /kaggle/input: {available}. '
        f'Da thu ca kagglehub handle: {SEADRONESSEE_DATASET_HANDLE}'
    )


DATA_ROOT = find_seadronessee_root()
CHECKPOINT_DATASET = 'nguyenducthangtb/echteai-seadronessee-convnext-fp32-baseline'

print('DATA_ROOT:', DATA_ROOT)
print('CHECKPOINT_DATASET:', CHECKPOINT_DATASET)

## 3. Cau hinh baseline

Cell nay co tinh chat quan trong nhat: no tao baseline `ConvNeXt backbone + Faster R-CNN default-style detector`.

Ban co the sua `FP32_TOTAL_EPOCHS` neu muon train dai hon.

In [ ]:
FP32_TOTAL_EPOCHS = 12
FP32_BATCH_SIZE_PER_GPU = 2
BENCHMARK_IMAGES = 100
TRAIN_LIMIT = None
UPLOAD_EVERY_EPOCH = True
EPOCHS_THIS_SESSION = 1

baseline = yaml.safe_load((REPO / 'configs/seadronessee_colab.yaml').read_text())
baseline['dataset'].update({
    'train_images': str(DATA_ROOT / 'images/train'),
    'train_annotations': str(DATA_ROOT / 'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT / 'images/val'),
    'val_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT / 'images/val'),
    'test_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'ignore_category_ids': [0],
    'num_classes': 6,
})

# Baseline detector theo huong mac dinh Faster R-CNN.
baseline['model'].update({
    'backbone': 'convnext_tiny',
    'pretrained_backbone': True,
    'trainable_backbone_layers': 4,
    'min_size': 800,
    'train_min_sizes': [800],
    'max_size': 1333,
    'anchor_sizes': [32, 64, 128, 256, 512],
    'aspect_ratios': [0.5, 1.0, 2.0],
    'use_focal_loss': False,
    'rpn_pre_nms_top_n_train': 2000,
    'rpn_pre_nms_top_n_test': 1000,
    'rpn_post_nms_top_n_train': 2000,
    'rpn_post_nms_top_n_test': 1000,
})

# Baseline train theo huong SGD cua Faster R-CNN reference.
baseline['training'].update({
    'fp32_batch_size': FP32_BATCH_SIZE_PER_GPU,
    'batch_size': FP32_BATCH_SIZE_PER_GPU,
    'fp32_epochs': FP32_TOTAL_EPOCHS,
    'epoch_benchmark_images': BENCHMARK_IMAGES,
    'optimizer': 'sgd',
    'fp32_lr': 0.005,
    'weight_decay': 0.0005,
    'lr_step_size': 8,
    'lr_gamma': 0.1,
    'warmup_iterations': 500,
    'print_frequency': 50,
})

baseline['output'] = {
    'directory': str(OUTPUT),
    'fp32_best': str(OUTPUT / 'fp32_best.pt'),
    'fp32_last': str(OUTPUT / 'fp32_last.pt'),
    'qat_best': str(OUTPUT / 'qat_best.pt'),
    'qat_last': str(OUTPUT / 'qat_last.pt'),
    'int8_model': str(OUTPUT / 'selective_int8.pt'),
    'evaluation_json': str(OUTPUT / 'evaluation.json'),
    'benchmark_json': str(OUTPUT / 'benchmark.json'),
    'epoch_benchmarks': str(OUTPUT / 'epoch_benchmarks.json'),
}

RUNTIME_CONFIG = WORK / 'runtime_baseline.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(baseline, sort_keys=False), encoding='utf-8')

print('Runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())

## 4. Helper cho log, train tung epoch va upload dataset version

In [ ]:
from datetime import datetime


def run_and_log(command, log_path, cwd):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(map(str, command)), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.now().isoformat(), flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f"\n===== START {datetime.now().isoformat()} =====\n")
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
        code = process.wait()
        log_file.write(f"===== END code={code} {datetime.now().isoformat()} =====\n")
    if code != 0:
        raise subprocess.CalledProcessError(code, command)


def checkpoint_epoch(path):
    path = Path(path)
    if not path.exists():
        return 0
    payload = torch.load(path, map_location='cpu', weights_only=False)
    return int(payload.get('epoch', 0)) if isinstance(payload, dict) else 0


def print_checkpoint_summary(directory):
    directory = Path(directory)
    for path in sorted(directory.glob('*')):
        if path.is_file():
            print(f'{path.name:40s} {path.stat().st_size / 2**20:9.2f} MB')


def upload_dataset_version(output_dir, dataset_handle, version_notes):
    output_dir = Path(output_dir)
    print(f'Uploading dataset version -> {dataset_handle}')
    print('Version notes:', version_notes)
    kagglehub.dataset_upload(dataset_handle, str(output_dir), version_notes=version_notes)
    print('Upload completed.')

## 5. Train baseline FP32 tung epoch va upload checkpoint moi epoch

Cell nay co the chay lai nhieu lan. Moi lan no se train tiep `EPOCHS_THIS_SESSION` epoch tu checkpoint `fp32_last.pt`.

Neu Kaggle co 2 GPU, script se tu dong dung DDP thong qua `train_next_epoch.py`.

In [ ]:
fp32_last = OUTPUT / 'fp32_last.pt'
fp32_best = OUTPUT / 'fp32_best.pt'

for _ in range(EPOCHS_THIS_SESSION):
    done = checkpoint_epoch(fp32_last)
    total = int(baseline['training']['fp32_epochs'])
    print('\n' + '=' * 80)
    print(f'Current FP32 epoch: {done}/{total}')
    if done >= total:
        print('FP32 baseline training already complete.')
        break

    command = [
        sys.executable, '-u', 'scripts/train_next_epoch.py',
        '--config', str(RUNTIME_CONFIG),
        '--stage', 'fp32',
    ]
    if fp32_last.exists():
        command += ['--resume', str(fp32_last)]
    if TRAIN_LIMIT is not None:
        command += ['--limit', str(TRAIN_LIMIT)]

    run_and_log(command, LOGS / f'fp32_epoch_{done + 1:02d}.log', cwd=REPO)

    saved_epoch = checkpoint_epoch(fp32_last)
    assert saved_epoch == done + 1, f'Expected epoch {done + 1}, got {saved_epoch}'

    epoch_copy = OUTPUT / f'fp32_epoch_{saved_epoch:02d}.pt'
    shutil.copy2(fp32_last, epoch_copy)
    if fp32_best.exists():
        shutil.copy2(fp32_best, OUTPUT / f'fp32_best_epoch_{saved_epoch:02d}.pt')

    print('Checkpoint summary after epoch', saved_epoch)
    print_checkpoint_summary(OUTPUT)

    if UPLOAD_EVERY_EPOCH:
        upload_dataset_version(
            OUTPUT,
            CHECKPOINT_DATASET,
            version_notes=f'SeaDronesSee FP32 baseline epoch {saved_epoch}/{total} with ConvNeXt backbone and default Faster R-CNN style settings',
        )

## 6. Helper danh gia final baseline: metrics, AP theo lop, benchmark, RAM/VRAM

In [ ]:
import psutil
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 2**20


@torch.inference_mode()
def collect_predictions(model, loader, device, progress_frequency=25):
    model.eval()
    predictions, targets = [], []
    processed = 0
    total = len(loader.dataset)
    print(f'collecting predictions: target={total} device={device}', flush=True)
    for images, batch_targets in loader:
        outputs = model([image.to(device) for image in images])
        predictions.extend([{key: value.detach().cpu() for key, value in output.items()} for output in outputs])
        targets.extend([{key: value.detach().cpu() if torch.is_tensor(value) else value for key, value in target.items()} for target in batch_targets])
        processed += len(images)
        if processed == 1 or processed % progress_frequency == 0 or processed >= total:
            print(f'prediction progress: {processed}/{total}', flush=True)
    return predictions, targets


def coco_per_class_ap(predictions, targets, dataset):
    results = []
    for prediction, target in zip(predictions, targets):
        image_id = int(target['image_id'])
        for box, label, score in zip(prediction['boxes'], prediction['labels'], prediction['scores']):
            x1, y1, x2, y2 = map(float, box)
            results.append({
                'image_id': image_id,
                'category_id': dataset.label_to_category_id[int(label)],
                'bbox': [x1, y1, x2 - x1, y2 - y1],
                'score': float(score),
            })
    coco_gt = COCO(str(dataset.annotation_path))
    valid_category_ids = sorted(dataset.category_id_to_label)
    coco_gt.dataset['categories'] = [c for c in coco_gt.dataset.get('categories', []) if int(c['id']) in valid_category_ids]
    coco_gt.dataset['annotations'] = [
        ann for ann in coco_gt.dataset.get('annotations', [])
        if int(ann['category_id']) in valid_category_ids and not ann.get('iscrowd', 0)
    ]
    for ann in coco_gt.dataset.get('annotations', []):
        ann.setdefault('iscrowd', 0)
        if 'area' not in ann:
            _, _, w, h = ann['bbox']
            ann['area'] = max(float(w), 0.0) * max(float(h), 0.0)
    coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(results) if results else coco_gt.loadRes([])
    evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
    evaluator.params.imgIds = [int(target['image_id']) for target in targets]
    evaluator.params.catIds = valid_category_ids
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    precision = evaluator.eval['precision']
    rows = []
    for class_index, category_id in enumerate(valid_category_ids):
        values = precision[:, :, class_index, 0, 2]
        valid = values[values > -1]
        ap = float(valid.mean()) if valid.size else float('nan')
        rows.append({
            'category_id': int(category_id),
            'label': int(dataset.category_id_to_label[category_id]),
            'class_name': dataset.label_to_name[dataset.category_id_to_label[category_id]],
            'AP': ap,
        })
    return pd.DataFrame(rows)


@torch.inference_mode()
def benchmark_gpu(model, loader, device, warmup_images=10, progress_frequency=25):
    model.eval()
    processed = 0
    timings = []
    total = len(loader.dataset)
    process = psutil.Process(os.getpid())
    rss_before = process.memory_info().rss / 2**20
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    started_all = time.perf_counter()
    for images, _ in loader:
        inputs = [image.to(device) for image in images]
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
        started = time.perf_counter()
        _ = model(inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
        elapsed_ms = (time.perf_counter() - started) * 1000.0 / max(len(inputs), 1)
        processed += len(inputs)
        if processed > warmup_images:
            timings.append(elapsed_ms)
        if processed == 1 or processed % progress_frequency == 0 or processed >= total:
            print(f'benchmark progress: {processed}/{total}', flush=True)
    rss_after = process.memory_info().rss / 2**20
    peak_vram_allocated = torch.cuda.max_memory_allocated(device) / 2**20 if device.type == 'cuda' else 0.0
    peak_vram_reserved = torch.cuda.max_memory_reserved(device) / 2**20 if device.type == 'cuda' else 0.0
    avg_ms = sum(timings) / max(len(timings), 1)
    return {
        'images': int(processed),
        'warmup_images': int(warmup_images),
        'measured_images': max(int(processed) - int(warmup_images), 0),
        'avg_inference_ms_per_image': float(avg_ms),
        'fps': 1000.0 / avg_ms if avg_ms > 0 else None,
        'device': str(device),
        'ram_before_mb': float(rss_before),
        'ram_after_mb': float(rss_after),
        'ram_delta_mb': float(max(rss_after - rss_before, 0.0)),
        'peak_vram_allocated_mb': float(peak_vram_allocated),
        'peak_vram_reserved_mb': float(peak_vram_reserved),
        'total_benchmark_seconds': float(time.perf_counter() - started_all),
    }

## 7. Danh gia baseline cuoi cung tren test split

Checkpoint duoc chon theo validation la `fp32_best.pt`, sau do test tren split test cua notebook nay (dang map vao validation annotations cua SeaDronesSee local setup).

In [ ]:
assert (OUTPUT / 'fp32_best.pt').exists(), 'Chua co fp32_best.pt; hay train baseline truoc'

eval_config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
device = choose_device('cuda' if torch.cuda.is_available() else 'cpu')
test_loader = build_coco_loader(eval_config, 'test', shuffle=False, limit=TRAIN_LIMIT, batch_size=1)
benchmark_loader = build_coco_loader(eval_config, 'test', shuffle=False, limit=BENCHMARK_IMAGES, batch_size=1)

model = build_fasterrcnn_convnext(eval_config)
load_checkpoint(OUTPUT / 'fp32_best.pt', model, map_location='cpu', strict=True)
model = model.to(device).eval()

predictions, targets = collect_predictions(model, test_loader, device, progress_frequency=50)
metrics = native_detection_metrics(predictions, targets)
dataset = unwrap_coco_dataset(test_loader.dataset)
canonical = _coco_metrics(predictions, targets, dataset)
if canonical:
    metrics.update(canonical)

print('Computing RPN recall via full evaluator ...')
rpn_metrics = evaluate_model(model, test_loader, device, include_rpn=True, progress_frequency=100)
for key, value in rpn_metrics.items():
    if key.startswith('rpn_') or key.startswith('proposal_'):
        metrics[key] = value

per_class_ap = coco_per_class_ap(predictions, targets, dataset)
benchmark = benchmark_gpu(model, benchmark_loader, device, warmup_images=10, progress_frequency=25)

report = {
    'stage': 'fp32_baseline_default_fasterrcnn',
    'checkpoint': str(OUTPUT / 'fp32_best.pt'),
    'checkpoint_epoch': checkpoint_epoch(OUTPUT / 'fp32_best.pt'),
    'checkpoint_size_mb': checkpoint_size_mb(OUTPUT / 'fp32_best.pt'),
    'model_state_size_mb': model_state_size_mb(model),
    'parameters': int(model.logical_parameter_count),
    'metrics': metrics,
    'benchmark': benchmark,
    'runtime_config': yaml.safe_load(RUNTIME_CONFIG.read_text()),
}

REPORT_JSON = REPORTS / 'fp32_baseline_report.json'
PER_CLASS_CSV = REPORTS / 'fp32_baseline_per_class_ap.csv'
REPORT_JSON.write_text(json.dumps(report, indent=2, allow_nan=True), encoding='utf-8')
per_class_ap.to_csv(PER_CLASS_CSV, index=False)

print(json.dumps(report, indent=2, allow_nan=True)[:5000])
display(per_class_ap)
print('Saved:', REPORT_JSON)
print('Saved:', PER_CLASS_CSV)

## 8. Anh minh hoa: phat hien dung, bo sot, phat hien nham

Cell nay luu mot loat anh minh hoa vao `visuals/`.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from torchvision.ops import box_iou


def tensor_to_pil(image_tensor):
    array = image_tensor.mul(255).clamp(0, 255).byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(array)


def draw_boxes(image, boxes, labels, scores=None, color='red', title=''):
    panel = image.copy()
    draw = ImageDraw.Draw(panel)
    font = ImageFont.load_default()
    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = map(float, box)
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        caption = str(labels[idx])
        if scores is not None:
            caption += f' {float(scores[idx]):.2f}'
        draw.text((x1 + 2, y1 + 2), caption, fill=color, font=font)
    if title:
        draw.rectangle((0, 0, panel.width, 20), fill='black')
        draw.text((5, 4), title, fill='white', font=font)
    return panel


@torch.inference_mode()
def collect_visual_cases(model, loader, device, score_threshold=0.5, max_images=12):
    cases = []
    for images, targets_batch in loader:
        outputs = model([images[0].to(device)])
        image = images[0]
        target = targets_batch[0]
        output = outputs[0]
        keep = output['scores'].detach().cpu() >= score_threshold
        pred_boxes = output['boxes'].detach().cpu()[keep]
        pred_labels = output['labels'].detach().cpu()[keep]
        pred_scores = output['scores'].detach().cpu()[keep]
        gt_boxes = target['boxes'].detach().cpu()
        gt_labels = target['labels'].detach().cpu()

        ious = box_iou(gt_boxes, pred_boxes) if len(gt_boxes) and len(pred_boxes) else torch.empty((len(gt_boxes), len(pred_boxes)))
        matched_gt = set()
        matched_pred = set()

        for gt_idx in range(len(gt_boxes)):
            if len(pred_boxes) == 0:
                continue
            values = ious[gt_idx]
            best_iou, best_idx = values.max(0)
            if best_iou >= 0.5 and int(gt_labels[gt_idx]) == int(pred_labels[best_idx]):
                matched_gt.add(gt_idx)
                matched_pred.add(int(best_idx))

        missed = [idx for idx in range(len(gt_boxes)) if idx not in matched_gt]
        false_positive = [idx for idx in range(len(pred_boxes)) if idx not in matched_pred]
        correct = [idx for idx in range(len(pred_boxes)) if idx in matched_pred]

        if correct:
            cases.append(('correct', image, gt_boxes, gt_labels, pred_boxes[correct], pred_labels[correct], pred_scores[correct]))
        if missed:
            cases.append(('missed', image, gt_boxes[missed], gt_labels[missed], pred_boxes, pred_labels, pred_scores))
        if false_positive:
            fp_idx = torch.tensor(false_positive, dtype=torch.long)
            cases.append(('false_positive', image, gt_boxes, gt_labels, pred_boxes[fp_idx], pred_labels[fp_idx], pred_scores[fp_idx]))
        if len(cases) >= max_images:
            break
    return cases[:max_images]


visual_loader = build_coco_loader(eval_config, 'test', shuffle=False, limit=100, batch_size=1)
cases = collect_visual_cases(model, visual_loader, device, score_threshold=0.5, max_images=12)
saved = []
for index, (kind, image_tensor, gt_boxes, gt_labels, pred_boxes, pred_labels, pred_scores) in enumerate(cases, 1):
    image = tensor_to_pil(image_tensor)
    gt_panel = draw_boxes(image, gt_boxes, gt_labels, color='lime', title='GT')
    pred_panel = draw_boxes(image, pred_boxes, pred_labels, pred_scores, color='red', title=f'Pred | {kind}')
    canvas = Image.new('RGB', (gt_panel.width * 2, gt_panel.height), 'white')
    canvas.paste(gt_panel, (0, 0))
    canvas.paste(pred_panel, (gt_panel.width, 0))
    out_path = VISUALS / f'{index:02d}_{kind}.jpg'
    canvas.save(out_path, quality=90)
    saved.append(out_path)

print('Saved visual samples:')
for path in saved:
    print(path)

## 9. Ve bieu do lich su baseline

In [ ]:
epoch_benchmarks = OUTPUT / 'epoch_benchmarks.json'
history = []
if epoch_benchmarks.exists():
    history = json.loads(epoch_benchmarks.read_text())

if history:
    df = pd.DataFrame(history)
    fp32_df = df[df['stage'] == 'fp32'].sort_values('epoch')
    display(fp32_df)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(fp32_df['epoch'], fp32_df['fps'], marker='o')
    axes[0].set_title('Epoch benchmark FPS')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True)

    axes[1].plot(fp32_df['epoch'], fp32_df['latency_ms_per_image'], marker='o', color='tab:red')
    axes[1].set_title('Epoch benchmark latency ms/image')
    axes[1].set_xlabel('Epoch')
    axes[1].grid(True)

    plt.tight_layout()
    plot_path = REPORTS / 'fp32_baseline_epoch_benchmark.png'
    fig.savefig(plot_path, dpi=160)
    print('Saved plot:', plot_path)
    plt.show()
else:
    print('No epoch benchmark history found yet.')

## 10. Upload artifact baseline cuoi cung len Kaggle Dataset

In [ ]:
required = [
    OUTPUT / 'fp32_best.pt',
    OUTPUT / 'fp32_last.pt',
    REPORTS / 'fp32_baseline_report.json',
    REPORTS / 'fp32_baseline_per_class_ap.csv',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing required artifacts:\n' + '\n'.join(missing)

print('Files in OUTPUT:')
print_checkpoint_summary(OUTPUT)
print('Files in REPORTS:')
print_checkpoint_summary(REPORTS)
print('Files in VISUALS:')
print_checkpoint_summary(VISUALS)

upload_dataset_version(
    WORK,
    CHECKPOINT_DATASET,
    version_notes='SeaDronesSee FP32 baseline final report, checkpoints, benchmark, per-class AP and visual samples',
)